In [1]:
import pandas as pd

from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

In [2]:
def drop_unnecessary(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=["native-country", "education", "marital-status", "relationship", "race", "sex"])

In [3]:
def occupation_mapping(occupation: str) -> str:
    if occupation.strip() in ["Adm-clerical", "Exec-managerial", "Prof-specialty", "Sales", "Tech-support"]:
        return "Office-Worker"
    elif occupation.strip() in ["Armed-Forces", "Protective-serv"]:
        return "Security-Worker"
    elif occupation.strip() in ["Craft-repair", "Machine-op-inspct", "Transport-moving"]:
        return "Machinery-Worker"
    elif occupation.strip() in ["Farming-fishing", "Handlers-cleaners", "Priv-house-serv"]:
        return "Manual-Worker"
    elif occupation.strip() == "Other-service":
        return "Other-Service"

In [4]:
def summarize_workclass(workclass: str) -> str:
    if workclass.strip() in ["Federal-gov", "Local-gov", "State-gov"]:
        return "GOV"
    elif workclass.strip() in ["Private", "Self-emp-inc", "Self-emp-not-inc"]:
        return "PRIVATE"
    else:
        return "UNPAID"

In [5]:
base_adult = pd.read_csv(filepath_or_buffer="adult\\adult.data")
base_adult

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [6]:
base_adult.drop(base_adult.loc[base_adult.workclass == " ?"].index, inplace=True)

In [7]:
base_adult.drop(base_adult.loc[base_adult.occupation == " ?"].index, inplace=True)

In [8]:
base_adult["occupation"] = base_adult["occupation"].apply(func=occupation_mapping)
base_adult

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Office-Worker,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Office-Worker,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Manual-Worker,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Manual-Worker,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Office-Worker,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Office-Worker,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machinery-Worker,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Office-Worker,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Office-Worker,Own-child,White,Male,0,0,20,United-States,<=50K


In [9]:
#base_adult["workclass"] = base_adult["workclass"].apply(func=summarize_workclass)
base_adult

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Office-Worker,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Office-Worker,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Manual-Worker,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Manual-Worker,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Office-Worker,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Office-Worker,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machinery-Worker,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Office-Worker,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Office-Worker,Own-child,White,Male,0,0,20,United-States,<=50K


In [10]:
base_adult = drop_unnecessary(df=base_adult)
base_adult

,age,workclass,fnlwgt,education-num,occupation,capital-gain,capital-loss,hours-per-week,income
0,39,State-gov,77516,13,Office-Worker,2174,0,40,<=50K
1,50,Self-emp-not-inc,83311,13,Office-Worker,0,0,13,<=50K
2,38,Private,215646,9,Manual-Worker,0,0,40,<=50K
3,53,Private,234721,7,Manual-Worker,0,0,40,<=50K
4,28,Private,338409,13,Office-Worker,0,0,40,<=50K
...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,12,Office-Worker,0,0,38,<=50K
32557,40,Private,154374,9,Machinery-Worker,0,0,40,>50K
32558,58,Private,151910,9,Office-Worker,0,0,40,<=50K
32559,22,Private,201490,9,Office-Worker,0,0,20,<=50K


In [11]:
X = base_adult.drop(columns=["income"])
y = base_adult["income"]

In [12]:
income_replacer = {" <=50K": 0, " >50K": 1}
y.replace(income_replacer, inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_42312\3940048061.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y.replace(income_replacer, inplace=True)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
X_train

,age,workclass,fnlwgt,education-num,occupation,capital-gain,capital-loss,hours-per-week
29030,38,Private,277248,14,Office-Worker,0,0,60
28231,39,Private,204527,16,Office-Worker,0,0,50
3888,46,Self-emp-not-inc,32825,9,Machinery-Worker,0,0,40
31926,29,Private,228346,11,Machinery-Worker,0,0,50
28553,25,Private,154941,13,Office-Worker,0,0,40
...,...,...,...,...,...,...,...,...
12975,90,Private,250832,6,Office-Worker,0,0,40
29622,51,Federal-gov,293196,10,Office-Worker,0,0,40
18459,44,Self-emp-inc,116358,13,Office-Worker,0,0,50
26391,29,State-gov,106972,14,Office-Worker,0,0,35


In [14]:
numeric_preprocessor = Pipeline(
    steps=[
        ("scaler", MinMaxScaler())
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("onehot", OneHotEncoder())
    ]
)

In [15]:
cat_variables = ["workclass", "occupation"]
num_variables = ["age", "fnlwgt", "education-num", "hours-per-week", "capital-loss", "capital-gain"]

In [16]:
knn_preprocessor = ColumnTransformer([
    ("categorical", categorical_preprocessor, cat_variables),
    ("numerical", numeric_preprocessor, num_variables)
])
knn_pipeline = make_pipeline(knn_preprocessor, KNeighborsClassifier())
knn_pipeline

,steps,"[('columntransformer', ...), ('kneighborsclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical', ...), ('numerical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
#knn_pipeline.fit(X_train, y_train)

In [18]:
knn_pipeline.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('categorical',
                                    Pipeline(steps=[('onehot', OneHotEncoder())]),
                                    ['workclass', 'occupation']),
                                   ('numerical',
                                    Pipeline(steps=[('scaler', MinMaxScaler())]),
                                    ['age', 'fnlwgt', 'education-num',
                                     'hours-per-week', 'capital-loss',
                                     'capital-gain'])])),
  ('kneighborsclassifier', KNeighborsClassifier())],
 'transform_input': None,
 'verbose': False,
 'columntransformer': ColumnTransformer(transformers=[('categorical',
                                  Pipeline(steps=[('onehot', OneHotEncoder())]),
                                  ['workclass', 'occupation']),
                                 ('numerical',
                                  Pipeline(steps=[('scaler'

In [21]:
knn_pipeline.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'columntransformer', 'kneighborsclassifier', 'columntransformer__force_int_remainder_cols', 'columntransformer__n_jobs', 'columntransformer__remainder', 'columntransformer__sparse_threshold', 'columntransformer__transformer_weights', 'columntransformer__transformers', 'columntransformer__verbose', 'columntransformer__verbose_feature_names_out', 'columntransformer__categorical', 'columntransformer__numerical', 'columntransformer__categorical__memory', 'columntransformer__categorical__steps', 'columntransformer__categorical__transform_input', 'columntransformer__categorical__verbose', 'columntransformer__categorical__onehot', 'columntransformer__categorical__onehot__categories', 'columntransformer__categorical__onehot__drop', 'columntransformer__categorical__onehot__dtype', 'columntransformer__categorical__onehot__feature_name_combiner', 'columntransformer__categorical__onehot__handle_unknown', 'columntransformer__categorical__o

In [19]:
param_grid = {
    "n_neighbors": [3, 5],
    "weights": ["uniform", "distance"]
}

In [20]:
search = GridSearchCV(knn_pipeline, param_grid, cv=5).fit(X_train, y_train)

ValueError: Invalid parameter 'n_neighbors' for estimator Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder())]),
                                                  ['workclass', 'occupation']),
                                                 ('numerical',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['age', 'fnlwgt',
                                                   'education-num',
                                                   'hours-per-week',
                                                   'capital-loss',
                                                   'capital-gain'])])),
                ('kneighb

In [ ]:
scores_knn = cross_val_score(search, X_test, y_test, cv=5)
scores_knn